# Figure 9.8: Large variance of slope (coefficients) of linear regression models (gray li…


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))


def linspace(a, b, n):
    return np.linspace(a, b, int(n))


def base_curve(x):
    return 1.0 + 0.7 * x - 0.35 * x**2 + 0.06 * x**3


def poly_design(x, degree):
    return np.vstack([x**k for k in range(degree + 1)]).T


def ridge_fit(x, y, degree, lam):
    X = poly_design(x, degree)
    I = np.eye(degree + 1)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def poly_predict(theta, x):
    X = poly_design(np.asarray(x), len(theta) - 1)
    return X @ theta


def line_fit(x, y, lam=0.0):
    X = np.column_stack([np.ones_like(x), x])
    I = np.eye(2)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def line_predict(theta, x):
    x = np.asarray(x)
    return theta[0] + theta[1] * x


def poly_data(seed=0, n=40, obs_noise=0.3, gt_noise=0.0):
    rng = rng_from_seed(seed)
    x = np.linspace(-3, 3, n)
    y_true = base_curve(x)
    y_star = y_true + gt_noise * rng.normal(size=n)
    y = y_star + obs_noise * rng.normal(size=n)
    return x, y, y_true, y_star

def draw(noise=0.35, trials=16, n=14, seed=7):
    xg = np.linspace(0.0, 2.0, 200)
    slopes = []
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    ax.plot(xg, 0.25 + 0.7 * xg, color="#2563eb", lw=3, label="true line")
    for t in range(trials):
        rng = rng_from_seed(seed * 100 + t + 1)
        x = np.sort(rng.uniform(0.0, 2.0, size=n))
        y = 0.25 + 0.7 * x + noise * rng.normal(size=n)
        th = line_fit(x, y, lam=0.0)
        slopes.append(th[1])
        ax.plot(xg, line_predict(th, xg), color="0.45", alpha=0.35, lw=2)
        ax.scatter(x, y, color="0.55", alpha=0.35, s=18)
    ax.set_title("Standard linear regression")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(alpha=0.2)
    plt.show()
    print(f"Slope mean = {np.mean(slopes):.3f}")
    print(f"Slope std  = {np.std(slopes):.3f}")

widgets.interact(
    draw,
    noise=widgets.FloatSlider(min=0.05, max=1.2, step=0.05, value=0.35, continuous_update=False),
    trials=widgets.IntSlider(min=5, max=40, step=1, value=16, continuous_update=False),
    n=widgets.IntSlider(min=8, max=30, step=1, value=14, continuous_update=False),
    seed=widgets.IntSlider(min=1, max=20, step=1, value=7, continuous_update=False),
)
